# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as a single object (not like a dict or list)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema organizes the dataset into `recordSet`s. Each `recordSet` contains `field`s (i.e., columns or features), each referenced by its unique `@id`.

In [ ]:
# Get all recordSet IDs from metadata
record_sets = dataset.metadata.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]

print("Available record sets and their @id:")
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '[No name]')}")

# Review fields of each record set
for rs in record_sets:
    print(f"\nFields for record set {rs['@id']}:")
    fields = rs.get('fields', [])
    for f in fields:
        print(f"  {f['@id']}: {f.get('name', '[No name]')}, dataType: {f.get('dataType','Unknown')}")

Let's examine the actual records from a record set. (Replace `<id_of_the_records_set>` below with actual `@id`.)

In [ ]:
# For demonstration purposes, choose the first record set
if record_set_ids:
    example_record_set_id = record_set_ids[0]

    print(f"Records from record set {example_record_set_id}:")
    for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
        print(x)
        if i > 2:  # Print just a few example records
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Fields are referenced by their `@id`.

This enables further downstream analysis using Python and pandas.

In [ ]:
# Extract data from all record sets into a dictionary of DataFrames
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

    print(f"Columns for record set {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
    print(dataframes[record_set_id].head())
    print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll demonstrate:
- Filtering by a numeric field (e.g., GAD-7 score)
- Normalizing numeric fields
- Grouping by a demographic attribute

All fields are referenced **by their `@id`**.

In [ ]:
# Example variables (replace with actual @id from the overview)
main_record_set_id = record_set_ids[0] if record_set_ids else None
# Let's try to find a numeric field (for demonstration, use GAD-7 score, PHQ-9 score, or PSQ score, if present)

# Find suitable numeric and group fields
df = dataframes[main_record_set_id]
numeric_field_candidates = [col for col in df.columns if 'score' in col.lower() or 'gad' in col.lower() or 'phq' in col.lower() or 'psq' in col.lower()]
if numeric_field_candidates:
    numeric_field_id = numeric_field_candidates[0]  # Use the first one found
else:
    numeric_field_id = df.select_dtypes(include='number').columns[0] if not df.empty and len(df.select_dtypes(include='number').columns) else None

# Find a group field (commonly demographic: age, gender, village, education, etc.)
group_field_candidates = [col for col in df.columns if any(word in col.lower() for word in ['age','gender','village','education'])]
if group_field_candidates:
    group_field_id = group_field_candidates[0]
else:
    group_field_id = df.columns[0] if not df.empty else None

print(f"Numeric field: {numeric_field_id}")
print(f"Group field: {group_field_id}")

# Apply EDA: filter by threshold
threshold = 10
if numeric_field_id is not None:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group and aggregate
    if group_field_id is not None and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

For example, plot the distribution of GAD-7 or PHQ-9 scores by demographic groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and group_field_id is not None:
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Plot distribution by group
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides mental health survey data from Kilifi County, Kenya, structured via Croissant schema.
- Record sets and fields can be referenced and extracted via their unique `@id`.
- Standard data science pipelines can be applied to filter, normalize, aggregate, and visualize data using `mlcroissant` and pandas.
- The dataset is suitable for deeper research and public health analysis.